# SIBI Dynamic Point-History Classifier (Phase 2F)

Trains an LSTM (following Kazuhito Takahashi's `point_history_classification` reference) that maps a 32-step wrist trajectory (32 × 2 = 64 floats) to one of the dynamic SIBI classes — word gestures (jeruk, kucing, besar, kecil, sangat) plus the motion-letters J and Z.

**Data shape**: each sample is 32 points sampled UNIFORMLY along a fixed 1.5-second window (wall-clock). The frontend recorder collects raw timestamped wrist positions over that window, then linear-interpolates them to 32 evenly-spaced output points. This decouples the trained model from the recording laptop's MediaPipe inference rate — a "jeruk" gesture has the same temporal shape whether collected on a 30fps machine or a 10fps one.

**Older CSVs collected before the time-resample migration** (raw 32 consecutive frames, variable wall-clock duration) are NOT compatible with this dataset and should be cleared / re-collected.

Input CSV: `point_history_csv/dynamic.csv` (collected via `/dev/gesture-recorder` or `frontend/public/recorder.html`).
Output: `point_history_classifier.hdf5`, `point_history_classifier.tflite`.

In [ ]:
import csv
import os
# Cap TF native threads BEFORE importing tf — improves stability on
# weaker CPUs. Required env vars are read only at first import.
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '4')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')  # silence INFO/WARN

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
FEATURE_LENGTH = 64  # 32 frames * 2 coords (must match DYNAMIC_HISTORY_SIZE in frontend recording/types.ts)
CSV_PATH = 'point_history_csv/dynamic_aug.csv'
LABEL_CSV_PATH = 'point_history_csv/dynamic_label.csv'
MODEL_PATH = 'point_history_classifier.keras'
# SavedModel dir is what convert_to_tfjs.sh consumes. We skip the .tflite
# intermediate because TFLiteConverter crashes on LSTM in some TF 2.16+
# envs — SavedModel → TFJS path doesn't hit that bug.
SAVED_MODEL_DIR = 'point_history_classifier_saved_model'

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total samples: {len(df)}')
print(f'Samples per class:\n{df["label"].value_counts()}')

# Derive label set + NUM_CLASSES from the actual data instead of a hardcoded
# constant — sorted for stable argmax-index mapping across runs. Writes back
# to LABEL_CSV_PATH so convert_to_tfjs.sh emits a labels.json that matches
# the trained model's softmax order.
labels = sorted(df['label'].astype(str).unique().tolist())
NUM_CLASSES = len(labels)
print(f'Derived {NUM_CLASSES} classes from data: {labels}')

pd.Series(labels).to_csv(LABEL_CSV_PATH, index=False, header=False)
print(f'Wrote {LABEL_CSV_PATH} with {NUM_CLASSES} class names.')

label_to_idx = {l: i for i, l in enumerate(labels)}
X = df.iloc[:, 1:].astype(np.float32).values
y_strings = df['label'].astype(str).values
y = np.array([label_to_idx[l] for l in y_strings], dtype=np.int32)
assert X.shape[1] == FEATURE_LENGTH, f'Expected {FEATURE_LENGTH} features, got {X.shape[1]}'
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# LSTM variant from Kazuhito's point_history_classification.ipynb.
# Reshape the flat (TIME_STEPS * DIMENSION,) feature vector into a proper
# (TIME_STEPS, DIMENSION) sequence so the LSTM can capture temporal order —
# critical for motion words like jeruk/kucing vs. their reverse-direction twins.
TIME_STEPS = 32  # = DYNAMIC_HISTORY_SIZE
DIMENSION = 2  # (x, y) per frame
assert TIME_STEPS * DIMENSION == FEATURE_LENGTH

model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(FEATURE_LENGTH,)),
    tf.keras.layers.Reshape((TIME_STEPS, DIMENSION), input_shape=(FEATURE_LENGTH,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(16, input_shape=[TIME_STEPS, DIMENSION]),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, verbose=1, save_weights_only=False, save_best_only=True,
)
# Smaller patience + epoch cap so a flaky env doesn't run for hours before
# any potential crash. LSTM on 305 train samples converges within ~200 epochs.
es_callback = tf.keras.callbacks.EarlyStopping(patience=500, verbose=1, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=1000,
    batch_size=64,           # smaller batch — less memory pressure
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback],
    verbose=2,
)

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred, target_names=labels, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('SIBI Dynamic Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Export as TF SavedModel for the TFJS converter. We skip the legacy
# TFLite intermediate because tf.lite.TFLiteConverter crashes on LSTM
# layers in some TF 2.16+ envs (segfault, no stack trace).
# tensorflowjs_converter --input_format=tf_saved_model reads this directly.
model.export(SAVED_MODEL_DIR)
print(f'Wrote SavedModel to {SAVED_MODEL_DIR}/')